
# 18 — Final Visuals Update With Multi-Indicator Validation  
## Final Report Visuals After Validation Synthesis

This notebook updates the final visual package after the validation synthesis.

It runs after:

```text
17_Validation_Synthesis_and_Final_Interpretation_2002_5Var.ipynb
```

## Purpose

Notebook 13 generated the initial final visuals.  
Notebook 18 adds final validation visuals based on:

- GDP recovery validation;
- multi-indicator macro validation;
- final validation synthesis;
- mismatch case summaries;
- final interpretation tables.

## Core framing

The final visuals support a **multidimensional structural POSet framework**.

They do not present:

```text
a final scalar Economic Resilience Index
```

## Main figures generated

- `18_final_validation_synthesis_dashboard.png`
- `18_gdp_vs_multi_indicator_validation_comparison.png`
- `18_macro_indicator_frontier_advantage_by_shock.png`
- `18_validation_mismatch_case_summary.png`
- `18_poset_incomparability_vs_validation_support.png`

## Main output tables

- `final_figure_inventory_updated.csv`
- `final_table_inventory_updated.csv`
- `final_visuals_update_run_summary.csv`
- `report_ready_figure_captions_updated.csv`
- `final_presentation_key_numbers.csv`


In [38]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

VALIDATION_SYNTHESIS_DIR = PROJECT_ROOT / "Data" / "Processed" / "Validation_Synthesis_Final_Interpretation"
MULTI_VALIDATION_DIR = PROJECT_ROOT / "Data" / "Processed" / "Multi_Indicator_Shock_Validation"
RECOVERY_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery_Validation"
PROFILE_DIR = PROJECT_ROOT / "Data" / "Processed" / "Profile_POSet_Main"
SENSITIVITY_DIR = PROJECT_ROOT / "Data" / "Processed" / "Sensitivity_Analysis"
FINAL_VISUALS_DIR = PROJECT_ROOT / "Data" / "Processed" / "Final_Visuals"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Final_Visuals_Update_With_Multi_Indicator_Validation"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "18_Final_Visuals_Update_With_Multi_Indicator_Validation"
FIGURES_DIR = PROJECT_ROOT / "Outputs" / "Figures" / "Final_Visuals_Update_With_Multi_Indicator_Validation"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Final_Visuals_Update_With_Multi_Indicator_Validation"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, FIGURES_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Figures folder:", FIGURES_DIR.resolve())


Run ID: 20260625_155224
Project root: C:\Documenti\UNIMIB\Data Science Lab\PROJECT\DS_01_ESROC_2026_06_25
Figures folder: C:\Documenti\UNIMIB\Data Science Lab\PROJECT\DS_01_ESROC_2026_06_25\Outputs\Figures\Final_Visuals_Update_With_Multi_Indicator_Validation


In [39]:

# ------------------------------------------------------
# CONFIGURATION AND INPUT FILES
# ------------------------------------------------------

INPUT_FILES = {
    "final_validation_table": VALIDATION_SYNTHESIS_DIR / "final_validation_table_updated.csv",
    "final_validation_synthesis": VALIDATION_SYNTHESIS_DIR / "final_validation_synthesis.csv",
    "final_interpretation_takeaways": VALIDATION_SYNTHESIS_DIR / "final_interpretation_takeaways.csv",
    "final_mismatch_case_summary": VALIDATION_SYNTHESIS_DIR / "final_mismatch_case_summary.csv",
    "validation_method_comparison": VALIDATION_SYNTHESIS_DIR / "validation_method_comparison.csv",
    "final_methodological_decisions": VALIDATION_SYNTHESIS_DIR / "final_methodological_decisions_updated.csv",
    "report_ready_final_paragraphs": VALIDATION_SYNTHESIS_DIR / "report_ready_final_interpretation_paragraphs.csv",

    "multi_evidence_matrix": MULTI_VALIDATION_DIR / "validation_evidence_matrix.csv",
    "multi_frontier_validation": MULTI_VALIDATION_DIR / "frontier_vs_nonfrontier_multi_indicator_validation.csv",
    "multi_takeaways": MULTI_VALIDATION_DIR / "multi_indicator_validation_takeaways.csv",
    "multi_mismatch_cases": MULTI_VALIDATION_DIR / "multi_indicator_validation_mismatch_cases.csv",

    "recovery_takeaways": RECOVERY_DIR / "recovery_validation_takeaway_table.csv",
    "profile_quality": PROFILE_DIR / "profile_poset_quality_diagnostics.csv",
    "robustness_dashboard": SENSITIVITY_DIR / "robustness_dashboard_table.csv",
}

REQUIRED_KEYS = [
    "final_validation_table",
    "multi_evidence_matrix",
    "multi_frontier_validation",
    "multi_takeaways",
    "recovery_takeaways",
    "profile_quality",
]

missing = [key for key in REQUIRED_KEYS if not INPUT_FILES[key].exists()]

if missing:
    raise FileNotFoundError(
        "Missing required input files:\n"
        + "\n".join([f"- {key}: {INPUT_FILES[key]}" for key in missing])
    )

MAIN_VARIABLE_SET = "baseline_5_variables"
MAIN_VARIABLE_COUNT = 5
MAIN_LEVELS = 5
TEMPORAL_DESIGN = "2007 and 2019 single pre-shock cross-sections"

print("Main variable set:", MAIN_VARIABLE_SET)
print("Main variable count:", MAIN_VARIABLE_COUNT)
print("Main levels:", MAIN_LEVELS)


Main variable set: baseline_5_variables
Main variable count: 5
Main levels: 5


In [40]:

# ------------------------------------------------------
# HELPERS
# ------------------------------------------------------

figure_inventory = []
table_inventory = []

def clean_keys(df):
    out = df.copy()

    if "Country" in out.columns:
        out["Country"] = out["Country"].astype(str).str.strip().str.upper()

    if "shock_id" in out.columns:
        out["shock_id"] = out["shock_id"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

    if "baseline_year" in out.columns:
        out["baseline_year"] = pd.to_numeric(out["baseline_year"], errors="coerce").astype("Int64")

    return out


def read_optional_csv(path):
    if path.exists():
        return clean_keys(pd.read_csv(path))
    return pd.DataFrame()


def save_table(df, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    df.to_csv(processed_path, index=False)
    df.to_csv(diagnostics_path, index=False)
    df.to_csv(table_path, index=False)

    table_inventory.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(df),
        "columns": len(df.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved table:", file_name)


def save_figure(fig, file_name, title, description):
    path = FIGURES_DIR / file_name
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    figure_inventory.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "path": str(path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved figure:", file_name)


def fmt_num(x, digits=2):
    if pd.isna(x):
        return "n/a"
    return f"{x:.{digits}f}"


In [41]:

# ------------------------------------------------------
# LOAD INPUTS
# ------------------------------------------------------

final_validation_table = read_optional_csv(INPUT_FILES["final_validation_table"])
final_validation_synthesis = read_optional_csv(INPUT_FILES["final_validation_synthesis"])
final_interpretation_takeaways = read_optional_csv(INPUT_FILES["final_interpretation_takeaways"])
final_mismatch_case_summary = read_optional_csv(INPUT_FILES["final_mismatch_case_summary"])
validation_method_comparison = read_optional_csv(INPUT_FILES["validation_method_comparison"])
final_methodological_decisions = read_optional_csv(INPUT_FILES["final_methodological_decisions"])
report_ready_final_paragraphs = read_optional_csv(INPUT_FILES["report_ready_final_paragraphs"])

multi_evidence_matrix = read_optional_csv(INPUT_FILES["multi_evidence_matrix"])
multi_frontier_validation = read_optional_csv(INPUT_FILES["multi_frontier_validation"])
multi_takeaways = read_optional_csv(INPUT_FILES["multi_takeaways"])
multi_mismatch_cases = read_optional_csv(INPUT_FILES["multi_mismatch_cases"])

recovery_takeaways = read_optional_csv(INPUT_FILES["recovery_takeaways"])
profile_quality = read_optional_csv(INPUT_FILES["profile_quality"])
robustness_dashboard = read_optional_csv(INPUT_FILES["robustness_dashboard"])

print("Loaded input shapes:")
for name, df in [
    ("final_validation_table", final_validation_table),
    ("final_validation_synthesis", final_validation_synthesis),
    ("multi_evidence_matrix", multi_evidence_matrix),
    ("multi_frontier_validation", multi_frontier_validation),
    ("multi_takeaways", multi_takeaways),
    ("recovery_takeaways", recovery_takeaways),
    ("profile_quality", profile_quality),
]:
    print(f"{name}: {df.shape}")

display(final_validation_table)
display(multi_takeaways)


Loaded input shapes:
final_validation_table: (2, 11)
final_validation_synthesis: (6, 9)
multi_evidence_matrix: (12, 7)
multi_frontier_validation: (12, 14)
multi_takeaways: (2, 8)
recovery_takeaways: (2, 8)
profile_quality: (2, 14)


,shock_id,countries,unique_profiles,frontier_countries,layers,incomparability_ratio,gdp_recovery_frontier_advantage_years,multi_indicator_frontier_better_metrics,multi_indicator_available_metrics,frontier_better_share,interpretation
0,2007,25,25,8,5,0.6800,0.8235,5,6,0.8333,Frontier countries show generally stronger pos...
1,2019,35,34,13,4,0.7701,0.2632,5,6,0.8333,Frontier countries show generally stronger pos...


,shock_id,available_metrics,frontier_better_metrics,nonfrontier_better_metrics,similar_metrics,frontier_better_share,headline,interpretation_caution
0,2007,6,5,1,0,0.8333,Frontier countries outperform non-frontier cou...,Descriptive validation only; no causal claim.
1,2019,6,5,1,0,0.8333,Frontier countries outperform non-frontier cou...,Descriptive validation only; no causal claim.


In [42]:

# ------------------------------------------------------
# FINAL PRESENTATION KEY NUMBERS TABLE
# ------------------------------------------------------

key_rows = []

for _, row in final_validation_table.iterrows():
    shock_id = str(row["shock_id"])

    key_rows.extend([
        {
            "shock_id": shock_id,
            "metric": "Countries",
            "value": row.get("countries"),
            "display_value": str(int(row.get("countries"))) if pd.notna(row.get("countries")) else "n/a",
            "interpretation": "Number of complete-case countries in the main POSet sample.",
        },
        {
            "shock_id": shock_id,
            "metric": "Unique profiles",
            "value": row.get("unique_profiles"),
            "display_value": str(int(row.get("unique_profiles"))) if pd.notna(row.get("unique_profiles")) else "n/a",
            "interpretation": "Observed unique five-variable ordinal profiles.",
        },
        {
            "shock_id": shock_id,
            "metric": "Frontier countries",
            "value": row.get("frontier_countries"),
            "display_value": str(int(row.get("frontier_countries"))) if pd.notna(row.get("frontier_countries")) else "n/a",
            "interpretation": "Countries in undominated structural profiles.",
        },
        {
            "shock_id": shock_id,
            "metric": "Incomparability ratio",
            "value": row.get("incomparability_ratio"),
            "display_value": fmt_num(row.get("incomparability_ratio")),
            "interpretation": "Share of profile pairs that cannot be ordered without arbitrary trade-offs.",
        },
        {
            "shock_id": shock_id,
            "metric": "GDP recovery frontier advantage",
            "value": row.get("gdp_recovery_frontier_advantage_years"),
            "display_value": f"{fmt_num(row.get('gdp_recovery_frontier_advantage_years'))} years",
            "interpretation": "Average recovery advantage of frontier countries over non-frontier countries.",
        },
        {
            "shock_id": shock_id,
            "metric": "Multi-indicator frontier-better share",
            "value": row.get("frontier_better_share"),
            "display_value": fmt_num(row.get("frontier_better_share")),
            "interpretation": "Share of macro validation indicators where frontier countries outperform non-frontier countries.",
        },
    ])

final_presentation_key_numbers = pd.DataFrame(key_rows)

save_table(
    final_presentation_key_numbers,
    "final_presentation_key_numbers.csv",
    "Final presentation key numbers",
    "Key values for final report/presentation slides.",
)

display(final_presentation_key_numbers)


Saved table: final_presentation_key_numbers.csv


,shock_id,metric,value,display_value,interpretation
0,2007,Countries,25.0000,25,Number of complete-case countries in the main ...
1,2007,Unique profiles,25.0000,25,Observed unique five-variable ordinal profiles.
2,2007,Frontier countries,8.0000,8,Countries in undominated structural profiles.
3,2007,Incomparability ratio,0.6800,0.68,Share of profile pairs that cannot be ordered ...
4,2007,GDP recovery frontier advantage,0.8235,0.82 years,Average recovery advantage of frontier countri...
5,2007,Multi-indicator frontier-better share,0.8333,0.83,Share of macro validation indicators where fro...
6,2019,Countries,35.0000,35,Number of complete-case countries in the main ...
7,2019,Unique profiles,34.0000,34,Observed unique five-variable ordinal profiles.
8,2019,Frontier countries,13.0000,13,Countries in undominated structural profiles.
9,2019,Incomparability ratio,0.7701,0.77,Share of profile pairs that cannot be ordered ...


In [93]:

# ------------------------------------------------------
# FIGURE 1 — FINAL VALIDATION SYNTHESIS DASHBOARD
# ------------------------------------------------------
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
})

dashboard = final_validation_table.copy()
dashboard["shock_id"] = dashboard["shock_id"].astype(str)

fig, ax = plt.subplots(figsize=(10, 5.8))
fig.subplots_adjust(bottom=0.22)  # keep room for the caption

x = np.arange(len(dashboard))
width = 0.25

# Consistent palette (blue for positive/supported, red for opposite; adjust if needed)
blue = "#1f77b4"
red = "#d62728"
navy = "#0B1F3B"
c_orange = "#ff7f0e"

ax.bar(x - width, dashboard["incomparability_ratio"], width=width,
       label="Incomparability ratio", color=navy)
ax.bar(x, dashboard["frontier_better_share"], width=width,
       label="Frontier-better share", color=blue)
ax.bar(
    x + width,
    dashboard["gdp_recovery_frontier_advantage_years"] / dashboard["gdp_recovery_frontier_advantage_years"].max(),
    width=width,
    label="GDP recovery advantage, scaled",
    color=red,
)

ax.set_xticks(x)
ax.set_xticklabels(dashboard["shock_id"], fontsize=13)
ax.set_ylim(0, 1.05)
ax.set_title("Final validation synthesis dashboard", fontweight="bold")
ax.set_xlabel("Shock")
ax.set_ylabel("Ratio / scaled value")
ax.legend()
ax.grid(True, axis="y", alpha=0.25)

ax.text(
    0.01, -0.18,
    "Higher frontier-better share supports validation; incomparability is a POSet feature, not a ranking failure.",
    transform=ax.transAxes,
    fontsize=10,
    ha="left",
    va="top",
)

save_figure(
    fig,
    "18_final_validation_synthesis_dashboard.png",
    "Final validation synthesis dashboard",
    "Combines POSet incomparability, multi-indicator validation share, and GDP recovery advantage.",
)


Saved figure: 18_final_validation_synthesis_dashboard.png


In [94]:
navy = "#0B1F3B"
red = "#D62728"

comparison = final_validation_table.copy()
comparison["shock_id"] = comparison["shock_id"].astype(str)

fig, ax = plt.subplots(figsize=(8, 5.5))
x = np.arange(len(comparison))
width = 0.35

ax.bar(
    x - width/2,
    comparison["gdp_recovery_frontier_advantage_years"],
    width=width,
    label="GDP recovery advantage, years",
    color=navy,
)

ax.bar(
    x + width/2,
    comparison["frontier_better_share"],
    width=width,
    label="Multi-indicator frontier-better share",
    color=red,
)

ax.set_xticks(x)
ax.set_xticklabels(comparison["shock_id"], fontsize=18, fontweight="bold")
ax.set_title("GDP recovery validation vs multi-indicator validation")
ax.set_xlabel("Shock")
ax.set_ylabel("Years / share")
ax.legend()
ax.grid(True, axis="y", alpha=0.25)

save_figure(
    fig,
    "18_gdp_vs_multi_indicator_validation_comparison.png",
    "GDP vs multi-indicator validation comparison",
    "Compares GDP recovery advantage with multi-indicator validation share.",
)


Saved figure: 18_gdp_vs_multi_indicator_validation_comparison.png


In [95]:
# ------------------------------------------------------
# FIGURE 3 — MACRO INDICATOR FRONTIER ADVANTAGE BY SHOCK
# ------------------------------------------------------

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

mpl.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 18,
    "axes.labelsize": 16,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
})

for shock_id, group in multi_frontier_validation.groupby("shock_id"):
    plot_df = group.copy().sort_values("frontier_advantage_direction_aligned")

    vals = plot_df["frontier_advantage_direction_aligned"].to_numpy()
    max_abs = float(max(abs(vals.min(initial=0)), abs(vals.max(initial=0))))

    # If all values are 0, avoid degenerate normalization
    if max_abs == 0:
        norm = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
        colors = plt.cm.RdBu(norm(vals))
    else:
        norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
        cmap = plt.cm.get_cmap("RdBu")  # red for negatives, blue for positives
        colors = cmap(norm(vals))

    fig, ax = plt.subplots(figsize=(8, 6))
    y = range(len(plot_df))

    ax.barh(list(y), vals, color=colors)
    ax.set_yticks(list(y))

    ax.set_yticklabels(
        plot_df["metric_label"],
        fontsize=13,
        fontweight="normal",
    )

    ax.axvline(0, linewidth=1)
    ax.set_title(
        f"Macro indicator frontier advantage — shock {shock_id}",
        fontweight="bold",
    )
    ax.set_xlabel("Frontier advantage, direction-aligned")
    ax.grid(True, axis="x", alpha=0.25)

    # Smaller, inside-axes annotation (so it doesn't dominate the plot)
    ax.text(
        0.01, -0.22,
        "Positive values mean frontier countries outperform non-frontier countries\n"
        "on the direction-aligned validation metric.",
        transform=ax.transAxes,
        fontsize=10,
        ha="left",
        va="top",
    )
    fig.subplots_adjust(bottom=0.25)
    
    save_figure(
        fig,
        f"18_macro_indicator_frontier_advantage_shock_{shock_id}.png",
        f"Macro indicator frontier advantage shock {shock_id}",
        "Direction-aligned frontier advantage by validation indicator.",
    )


Saved figure: 18_macro_indicator_frontier_advantage_shock_2007.png
Saved figure: 18_macro_indicator_frontier_advantage_shock_2019.png


In [96]:

# ------------------------------------------------------
# FIGURE 4 — VALIDATION MISMATCH CASE SUMMARY
# ------------------------------------------------------

if len(final_mismatch_case_summary):
    mismatch_plot = final_mismatch_case_summary.copy()
    mismatch_plot["label"] = mismatch_plot["shock_id"].astype(str) + "\n" + mismatch_plot["mismatch_source"].astype(str)

    fig, ax = plt.subplots(figsize=(10, 5.8))

    ax.bar(mismatch_plot["label"], mismatch_plot["mismatch_cases"])
    ax.set_title("Validation mismatch case summary")
    ax.set_xlabel("Shock and validation layer")
    ax.set_ylabel("Mismatch cases")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, axis="y", alpha=0.25)

    save_figure(
        fig,
        "18_validation_mismatch_case_summary.png",
        "Validation mismatch case summary",
        "Counts of mismatch cases by shock and validation layer.",
    )
else:
    final_mismatch_case_summary = pd.DataFrame(columns=["shock_id", "mismatch_source", "mismatch_cases"])

save_table(
    final_mismatch_case_summary,
    "final_visual_mismatch_case_summary.csv",
    "Final mismatch case summary",
    "Mismatch case counts used for final visual package.",
)

display(final_mismatch_case_summary)


Saved figure: 18_validation_mismatch_case_summary.png
Saved table: final_visual_mismatch_case_summary.csv


,shock_id,mismatch_source,mismatch_cases,case_type_summary,interpretation
0,2007,GDP recovery validation,4,frontier_but_slow_recovery,GDP recovery mismatches identify cases where s...
1,2019,GDP recovery validation,2,frontier_but_slow_recovery,GDP recovery mismatches identify cases where s...
2,2007,Multi-indicator validation,22,"deep_layer_but_stronger_validation_outcome, fr...",Multi-indicator mismatches show metric-specifi...
3,2019,Multi-indicator validation,41,"deep_layer_but_stronger_validation_outcome, fr...",Multi-indicator mismatches show metric-specifi...


In [97]:

# ------------------------------------------------------
# FIGURE 5 — POSET INCOMPARABILITY VS VALIDATION SUPPORT
# ------------------------------------------------------

plot_df = final_validation_table.copy()

fig, ax = plt.subplots(figsize=(8, 5.5))

ax.scatter(
    plot_df["incomparability_ratio"],
    plot_df["frontier_better_share"],
    s=160,
)

for _, row in plot_df.iterrows():
    ax.text(
        row["incomparability_ratio"],
        row["frontier_better_share"],
        str(row["shock_id"]),
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_title("POSet incomparability vs validation support")
ax.set_xlabel("Incomparability ratio")
ax.set_ylabel("Frontier-better validation share")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.25)

save_figure(
    fig,
    "18_poset_incomparability_vs_validation_support.png",
    "POSet incomparability vs validation support",
    "Compares structural incomparability with multi-indicator validation support.",
)


Saved figure: 18_poset_incomparability_vs_validation_support.png


In [98]:

# ------------------------------------------------------
# REPORT-READY FIGURE CAPTIONS
# ------------------------------------------------------

report_ready_figure_captions_updated = pd.DataFrame([
    {
        "figure_file": "18_final_validation_synthesis_dashboard.png",
        "caption": (
            "Final validation synthesis dashboard combining POSet incomparability, multi-indicator validation support, "
            "and GDP recovery advantage. The figure summarizes validation evidence without converting the POSet into a scalar index."
        ),
    },
    {
        "figure_file": "18_gdp_vs_multi_indicator_validation_comparison.png",
        "caption": (
            "Comparison between GDP recovery validation and broader multi-indicator validation. Frontier countries show "
            "faster recovery on average and outperform non-frontier countries on most macro validation indicators."
        ),
    },
    {
        "figure_file": "18_macro_indicator_frontier_advantage_shock_2007.png / 18_macro_indicator_frontier_advantage_shock_2019.png",
        "caption": (
            "Direction-aligned frontier advantage by macro validation indicator. Positive values mean frontier countries "
            "perform better than non-frontier countries on the given post-shock outcome."
        ),
    },
    {
        "figure_file": "18_validation_mismatch_case_summary.png",
        "caption": (
            "Mismatch cases by validation layer and shock. These cases are expected in a multidimensional framework and "
            "indicate where post-shock outcomes diverge from structural POSet position."
        ),
    },
    {
        "figure_file": "18_poset_incomparability_vs_validation_support.png",
        "caption": (
            "Relationship between POSet incomparability and validation support. High incomparability shows why a scalar "
            "ranking would hide structural trade-offs."
        ),
    },
])

save_table(
    report_ready_figure_captions_updated,
    "report_ready_figure_captions_updated.csv",
    "Report-ready figure captions updated",
    "Final report captions for validation-synthesis figures.",
)

display(report_ready_figure_captions_updated)


Saved table: report_ready_figure_captions_updated.csv


,figure_file,caption
0,18_final_validation_synthesis_dashboard.png,Final validation synthesis dashboard combining...
1,18_gdp_vs_multi_indicator_validation_compariso...,Comparison between GDP recovery validation and...
2,18_macro_indicator_frontier_advantage_shock_20...,Direction-aligned frontier advantage by macro ...
3,18_validation_mismatch_case_summary.png,Mismatch cases by validation layer and shock. ...
4,18_poset_incomparability_vs_validation_support...,Relationship between POSet incomparability and...


In [99]:

# ------------------------------------------------------
# FINAL UPDATED INVENTORIES AND AUDIT WORKBOOK
# ------------------------------------------------------

final_figure_inventory_updated = pd.DataFrame(figure_inventory)
final_table_inventory_updated = pd.DataFrame(table_inventory)

final_visuals_update_run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "created_at": RUN_TIMESTAMP,
    "main_variable_set": MAIN_VARIABLE_SET,
    "main_variable_count": MAIN_VARIABLE_COUNT,
    "main_levels": MAIN_LEVELS,
    "temporal_design": TEMPORAL_DESIGN,
    "figures_created": len(final_figure_inventory_updated),
    "tables_created": len(final_table_inventory_updated),
    "validation_figures_added": len(final_figure_inventory_updated),
    "note": "Final validation visuals update. Does not create a scalar index.",
}])

for table, file_name, title, description in [
    (
        final_figure_inventory_updated,
        "final_figure_inventory_updated.csv",
        "Final figure inventory updated",
        "Inventory of final validation-synthesis visuals.",
    ),
    (
        final_table_inventory_updated,
        "final_table_inventory_updated.csv",
        "Final table inventory updated",
        "Inventory of final validation-synthesis tables.",
    ),
    (
        final_visuals_update_run_summary,
        "final_visuals_update_run_summary.csv",
        "Final visuals update run summary",
        "Run summary for validation-synthesis visual update.",
    ),
]:
    table.to_csv(PROCESSED_DIR / file_name, index=False)
    table.to_csv(DIAGNOSTICS_DIR / file_name, index=False)
    table.to_csv(TABLES_DIR / file_name, index=False)

audit_xlsx = AUDIT_DIR / "18_Final_Visuals_Update_With_Multi_Indicator_Validation_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    final_visuals_update_run_summary.to_excel(writer, sheet_name="run_summary", index=False)
    final_figure_inventory_updated.to_excel(writer, sheet_name="figure_inventory", index=False)
    final_table_inventory_updated.to_excel(writer, sheet_name="table_inventory", index=False)
    final_presentation_key_numbers.to_excel(writer, sheet_name="key_numbers", index=False)
    final_validation_table.to_excel(writer, sheet_name="final_validation_table", index=False)
    multi_frontier_validation.to_excel(writer, sheet_name="macro_advantage", index=False)
    final_mismatch_case_summary.to_excel(writer, sheet_name="mismatch_summary", index=False)
    report_ready_figure_captions_updated.to_excel(writer, sheet_name="figure_captions", index=False)
    final_interpretation_takeaways.to_excel(writer, sheet_name="takeaways", index=False)

print("Audit workbook:", audit_xlsx.resolve())
display(final_visuals_update_run_summary)
display(final_figure_inventory_updated)
display(final_table_inventory_updated)


ModuleNotFoundError: No module named 'xlsxwriter'

In [ ]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

expected_figures = [
    "18_final_validation_synthesis_dashboard.png",
    "18_gdp_vs_multi_indicator_validation_comparison.png",
    "18_macro_indicator_frontier_advantage_shock_2007.png",
    "18_macro_indicator_frontier_advantage_shock_2019.png",
    "18_validation_mismatch_case_summary.png",
    "18_poset_incomparability_vs_validation_support.png",
]

figure_check = pd.DataFrame([
    {
        "figure_file": f,
        "exists": (FIGURES_DIR / f).exists(),
        "path": str(FIGURES_DIR / f),
    }
    for f in expected_figures
])

expected_tables = [
    "final_presentation_key_numbers.csv",
    "final_visual_macro_indicator_frontier_advantage.csv",
    "final_visual_mismatch_case_summary.csv",
    "report_ready_figure_captions_updated.csv",
    "final_figure_inventory_updated.csv",
    "final_table_inventory_updated.csv",
    "final_visuals_update_run_summary.csv",
]

table_check = pd.DataFrame([
    {
        "table_file": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_tables
])

figure_check.to_csv(DIAGNOSTICS_DIR / "figure_output_check.csv", index=False)
table_check.to_csv(DIAGNOSTICS_DIR / "table_output_check.csv", index=False)

print("18 FINAL VISUALS UPDATE COMPLETE")
print("=" * 90)

display(figure_check)
display(table_check)

print("\nImportant final framing:")
print("1. Visuals support a multidimensional structural POSet framework.")
print("2. They do not convert the analysis into a scalar Economic Resilience Index.")
print("3. Validation evidence is descriptive, not causal.")

print("\nKey outputs to inspect/send:")
print("- 18_Final_Visuals_Update_With_Multi_Indicator_Validation_Audit.xlsx")
print("- final_figure_inventory_updated.csv")
print("- final_table_inventory_updated.csv")
print("- report_ready_figure_captions_updated.csv")
print("- final_presentation_key_numbers.csv")

print("\nNext step:")
print("Prepare final report/slides package.")


18 FINAL VISUALS UPDATE COMPLETE


,figure_file,exists,path
0,18_final_validation_synthesis_dashboard.png,True,D:\Milano Bicocca\Course Materials\1st Year\02...
1,18_gdp_vs_multi_indicator_validation_compariso...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
2,18_macro_indicator_frontier_advantage_shock_20...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
3,18_macro_indicator_frontier_advantage_shock_20...,True,D:\Milano Bicocca\Course Materials\1st Year\02...
4,18_validation_mismatch_case_summary.png,True,D:\Milano Bicocca\Course Materials\1st Year\02...
5,18_poset_incomparability_vs_validation_support...,True,D:\Milano Bicocca\Course Materials\1st Year\02...


,table_file,processed_exists,diagnostics_exists,tables_exists
0,final_presentation_key_numbers.csv,True,True,True
1,final_visual_macro_indicator_frontier_advantag...,True,True,True
2,final_visual_mismatch_case_summary.csv,True,True,True
3,report_ready_figure_captions_updated.csv,True,True,True
4,final_figure_inventory_updated.csv,True,True,True
5,final_table_inventory_updated.csv,True,True,True
6,final_visuals_update_run_summary.csv,True,True,True



Important final framing:
1. Visuals support a multidimensional structural POSet framework.
2. They do not convert the analysis into a scalar Economic Resilience Index.
3. Validation evidence is descriptive, not causal.

Key outputs to inspect/send:
- 18_Final_Visuals_Update_With_Multi_Indicator_Validation_Audit.xlsx
- final_figure_inventory_updated.csv
- final_table_inventory_updated.csv
- report_ready_figure_captions_updated.csv
- final_presentation_key_numbers.csv

Next step:
Prepare final report/slides package.
